# Milestone 3 — Model Building (ARIMA + GJR-GARCH vs LSTM)
**Project:** Stock Market Trend Analysis | **Seed:** 42

## What this notebook does

M1 built the leakage-safe dataset; M2 showed returns are **mean-stationary but conditionally
heteroskedastic with a leverage effect**, with **weak linear next-day mean predictability** but a real
**nonlinear volatility signal** (mutual information). M3 turns that into models:

1. **Baselines** — naive-zero, persistence, moving-average. The bar every model must clear.
2. **Classical** — per-ticker **ARIMA** on the return mean + **GJR-GARCH (Student-t)** for the variance
   (M2's leverage effect → an *asymmetric* GARCH; GJR is the implemented variant — EGARCH is an alternative).
3. **Deep** — one **global LSTM** on the 30 `MODEL_FEATURES` (ticker one-hot). The MI evidence says the
   exploitable signal is nonlinear + volatility-based → this is what the LSTM is for.
4. **Compare** on `val`, pick a winner, then **touch `holdout` exactly once** at the very end.

**Metrics:** RMSE / MAE on next-day log returns + **directional accuracy** (sign hit-rate). For this domain a
52–55% hit rate is the realistic, meaningful target — a low RMSE alone means little.

**Data discipline:** train on `train_fe`, tune on `val_fe`, `holdout_fe` stays sealed until §8.
Target = `target_log_return` (next-day). Train on `MODEL_FEATURES` only; drop warm-up NaN rows.

> **LSTM note:** this notebook runs a tiny **2-epoch CPU smoke-test** locally to prove the Keras code works.
> Full training (20 epochs, early stopping) is meant to run in **Google Colab** (GPU). The smoke-test RMSE is
> NOT the real LSTM score — it is only a "does it run" check; the Colab run fills the real number.

## 0. Setup

In [ ]:
import sys
import subprocess

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from pathlib import Path as _P

    _req = _P("/content/Stock-Market-Trend-Analysis/requirements.txt")
    if _req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)], check=False)

# Ensure the M3-specific libraries are present (local or Colab).
for pkg, mod in [("arch", "arch"), ("tensorflow", "tensorflow"), ("optuna==4.1.0", "optuna")]:
    try:
        __import__(mod)
    except ImportError:
        # Local fallback uses the CPU build; Colab already ships tensorflow.
        target = "tensorflow-cpu==2.18.0" if (pkg == "tensorflow" and not IN_COLAB) else pkg
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=False)

In [ ]:
import json
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")  # statsmodels/arch convergence chatter

SEED = 42
np.random.seed(SEED)

if IN_COLAB:
    ROOT = Path("/content/Stock-Market-Trend-Analysis")
elif "__file__" in globals():
    ROOT = Path(__file__).resolve().parent.parent
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROC = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
MODELS = ROOT / "models"
FIG.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

# M1 outputs (train_fe/val_fe/feature_roles/...) may be absent in a fresh/partial session. Regenerate them
# once by running M1, which reads the committed data/raw/*.csv snapshots (cache-first, no network).
if not (PROC / "feature_roles.json").exists():
    _m1 = ROOT / "notebooks" / "01_data_collection_preprocessing.py"
    print(f"feature_roles.json missing -> running M1 to regenerate it ({_m1.name})...")
    subprocess.run([sys.executable, str(_m1)], check=True, cwd=str(ROOT))

roles = json.loads((PROC / "feature_roles.json").read_text())
MODEL_FEATURES = roles["model_features"]
CANDIDATE_FEATURES = roles.get("candidate_features", [])  # M3.5 ablation pool (gated into X by section 1b)
TARGET = roles["target"]
TICKERS = ["^GSPC", "AAPL", "AMZN", "NVDA"]

# Load train + val (allowed in M3). Holdout stays sealed until section 8.
train = pd.read_parquet(PROC / "train_fe.parquet")
val = pd.read_parquet(PROC / "val_fe.parquet")
for d in (train, val):
    d["date"] = pd.to_datetime(d["date"])

print("train:", train.shape, "| val:", val.shape)
print("target:", TARGET, "| n features:", len(MODEL_FEATURES))

## 1. Metrics

Three metrics, unit-tested on a toy array so we trust them.

In [ ]:
def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))


def directional_accuracy(y_true, y_pred):
    """Fraction of days where the predicted SIGN matches the actual sign (zeros excluded).
    Returns NaN if the prediction has no directional opinion (all preds 0)."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    if np.all(y_pred == 0):
        return float("nan")  # a zero-prediction model has NO directional view -> n/a, not 0.0
    mask = y_true != 0
    return float(np.mean(np.sign(y_pred[mask]) == np.sign(y_true[mask])))


def diracc_pvalue(y_true, y_pred):
    """Binomial test that directional accuracy != 0.50 (is the sign hit-rate better than a coin flip?)."""
    from scipy.stats import binomtest
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    if np.all(y_pred == 0):
        return float("nan")
    mask = y_true != 0
    hits = int(np.sum(np.sign(y_pred[mask]) == np.sign(y_true[mask])))
    n = int(mask.sum())
    return float(binomtest(hits, n, 0.5).pvalue)


def diebold_mariano(y_true, pred_a, pred_b):
    """Diebold-Mariano test (squared-error loss) that model A and B have EQUAL accuracy.
    Returns (DM stat, p-value). p<0.05 => the RMSE difference is statistically real, not noise."""
    from scipy.stats import norm
    y_true = np.asarray(y_true, float)
    da = (y_true - np.asarray(pred_a, float)) ** 2
    db = (y_true - np.asarray(pred_b, float)) ** 2
    d = da - db
    dbar = d.mean()
    n = len(d)
    var = np.var(d, ddof=1)
    if var == 0:
        return 0.0, 1.0
    dm = dbar / np.sqrt(var / n)
    return float(dm), float(2 * (1 - norm.cdf(abs(dm))))


# Toy unit tests
assert abs(rmse([0, 0, 0], [1, 1, 1]) - 1.0) < 1e-9
assert abs(rmse([0, 0], [3, 4]) - np.sqrt(12.5)) < 1e-9
assert abs(mae([1, 2, 3], [1, 2, 3])) < 1e-9
assert abs(directional_accuracy([1, -1, 1, -1], [2, -3, -1, -1]) - 0.75) < 1e-9
assert np.isnan(directional_accuracy([1, -1], [0, 0]))
print("metric unit tests passed")
_METRICS_TESTED = True


def score(name, y_true, y_pred):
    da = directional_accuracy(y_true, y_pred)
    return {"model": name, "RMSE": rmse(y_true, y_pred), "MAE": mae(y_true, y_pred),
            "DirAcc": da, "DirAcc_p": diracc_pvalue(y_true, y_pred)}


def clean_xy(df):
    """Per-split frame -> (rows with non-NaN target & features), sorted by ticker/date.
    Drops over MODEL_FEATURES + CANDIDATE_FEATURES so the ablation pool is fully populated; because the
    longest candidate warm-up (semi_vol_63) is <= the existing realized_vol_63 warm-up, this loses 0 extra rows.
    The assert makes that claim LOUD — if a candidate ever NaNs outside the MF warm-up, fail instead of silently
    dropping an is_warmup=0 row the nan-report never counted."""
    base = df.dropna(subset=[TARGET] + MODEL_FEATURES)
    d = df.dropna(subset=[TARGET] + MODEL_FEATURES + CANDIDATE_FEATURES)
    assert len(base) == len(d), f"candidate warm-up dropped {len(base) - len(d)} extra rows (silent drop!)"
    return d.sort_values(["ticker", "date"]).reset_index(drop=True)


train_c = clean_xy(train)
val_c = clean_xy(val)
print("after dropna -> train:", train_c.shape, "val:", val_c.shape)

## 1b. Feature ablation — do the candidate variance-axis features earn a place in X?

M2 established the predictable structure is **variance** (`parkinson_vol_21` MI 0.16 vs a 0.002 noise floor),
not **direction** (Pearson |r| < 0.075 for every feature). So in the M3.5 hardening pass we added a small pool
of variance-axis CANDIDATE features (`semi_vol_21/63`, `ret_skew_21`, `ret_kurt_21`, `atr_14`) and let a
**train-only** ablation decide which — if any — enter `X`. Nothing here reads val or holdout: selection on the
test set is leakage, so the whole harness lives inside `train_c`.

Two independent train-only signals, and a candidate must clear **both**:
1. **Mutual information** vs the next-day target, above a shuffled-target noise floor (+3 sigma).
2. **Walk-forward permutation importance** — train LightGBM on expanding folds, permute each candidate on the
   out-of-sample block, and keep only those whose permutation *raises* OOS RMSE on average (i.e. the model
   actually relies on them beyond the 39 baseline features).

The `date_folds` helper below (5 expanding folds, split by DATE so all tickers share the boundary, purged by
one trading day so the next-day label can't bleed across a fold edge) is reused by the Optuna tuning in 5b.

In [ ]:
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression

CV_OOS_YEARS = [2019, 2020, 2021, 2022, 2023]  # 5 expanding walk-forward folds, all inside TRAIN


def date_folds(df, oos_years=CV_OOS_YEARS, embargo=1):
    """Purged expanding walk-forward folds over TRAIN only. Yields (train_idx, oos_idx) into df.index:
    expanding train history, one OOS calendar year each, split by DATE (all tickers share the threshold),
    purged by `embargo` trading days at the train's right edge so the last train row's next-day target
    (which lands in the OOS block) is removed -- the only label overlap in a 1-step-ahead expanding scheme."""
    d = df.sort_values(["ticker", "date"])
    dates = d["date"]
    for y in oos_years:
        oos_start, oos_end = pd.Timestamp(y, 1, 1), pd.Timestamp(y + 1, 1, 1)
        oos_mask = (dates >= oos_start) & (dates < oos_end)
        if not oos_mask.any():
            continue
        tr_dates = dates[dates < oos_start].drop_duplicates().sort_values()
        if len(tr_dates) <= embargo:
            continue
        cut = tr_dates.iloc[-embargo]          # purge the last `embargo` train dates
        yield d.index[dates < cut], d.index[oos_mask]


_folds = list(date_folds(train_c))
print(f"walk-forward folds (train-only): {len(_folds)}")
for i, (tr, oo) in enumerate(_folds, 1):
    print(f"  fold {i}: train {len(tr):5d} -> OOS {len(oo):5d} "
          f"[{train_c.loc[oo, 'date'].min().date()}..{train_c.loc[oo, 'date'].max().date()}]")

_rng = np.random.RandomState(SEED)
POOL = MODEL_FEATURES + CANDIDATE_FEATURES

# Signal 1: mutual information vs next-day target + shuffled-target noise floor (+3 sigma).
mi = pd.Series(mutual_info_regression(train_c[POOL].values, train_c[TARGET].values, random_state=SEED), index=POOL)
_floor_samples = [mutual_info_regression(train_c[CANDIDATE_FEATURES].values, _rng.permutation(train_c[TARGET].values),
                                         random_state=SEED) for _ in range(5)]
MI_FLOOR = float(np.mean(_floor_samples)) + 3 * float(np.std(_floor_samples))
print(f"\nMI noise floor (shuffled target, +3 sigma): {MI_FLOOR:.5f}")


# Signal 2: walk-forward permutation importance (mean OOS RMSE increase when a candidate is shuffled).
def _wf_perm_importance(pool, candidates):
    params = dict(objective="regression", n_estimators=300, learning_rate=0.03, num_leaves=31,
                  min_child_samples=50, subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
                  random_state=SEED, n_jobs=-1, verbosity=-1, deterministic=True, force_row_wise=True)
    feat_t = pool + ["ticker"]
    deltas = {c: [] for c in candidates}
    for tr_idx, oo_idx in date_folds(train_c):
        Xtr = train_c.loc[tr_idx, pool].copy(); Xtr["ticker"] = train_c.loc[tr_idx, "ticker"].astype("category")
        Xoo = train_c.loc[oo_idx, pool].copy(); Xoo["ticker"] = train_c.loc[oo_idx, "ticker"].astype("category")
        ytr, yoo = train_c.loc[tr_idx, TARGET].values, train_c.loc[oo_idx, TARGET].values
        m = lgb.LGBMRegressor(**params).fit(Xtr[feat_t], ytr, categorical_feature=["ticker"])
        base = rmse(yoo, m.predict(Xoo[feat_t]))
        for c in candidates:
            Xp = Xoo.copy(); Xp[c] = _rng.permutation(Xp[c].values)
            deltas[c].append(rmse(yoo, m.predict(Xp[feat_t])) - base)
    return {c: (float(np.mean(v)), float(np.std(v))) for c, v in deltas.items()}


_perm = _wf_perm_importance(POOL, CANDIDATE_FEATURES)

# KEEP rule: clears the MI floor AND its walk-forward permutation importance EXCEEDS ITS OWN CROSS-FOLD NOISE
# (perm_dRMSE > perm_std) — a mean that is merely positive-by-a-hair is indistinguishable from zero and would
# keep a feature on noise, so we require the permutation benefit to be at least one fold-std above zero.
_abl_rows, KEPT = [], []
for c in CANDIDATE_FEATURES:
    mi_c, (p_mean, p_std) = float(mi[c]), _perm[c]
    keep = (mi_c > MI_FLOOR) and (p_mean > p_std)
    KEPT.append(c) if keep else None
    _abl_rows.append({"candidate": c, "MI": round(mi_c, 5), "MI_floor": round(MI_FLOOR, 5),
                      "perm_dRMSE": round(p_mean, 7), "perm_std": round(p_std, 7),
                      "verdict": "KEEP" if keep else "reject"})
abl_table = pd.DataFrame(_abl_rows)
print("\nAblation verdicts (train-only; val/holdout never touched):")
print(abl_table.to_string(index=False))
abl_table.to_csv(MODELS / "feature_ablation.csv", index=False)

FEAT_MODEL = MODEL_FEATURES + KEPT   # the ACTIVE feature set for every downstream model (LGB + LSTM)
print(f"\nKEPT candidates: {KEPT if KEPT else 'NONE'}")
print(f"FEAT_MODEL = {len(MODEL_FEATURES)} baseline + {len(KEPT)} kept = {len(FEAT_MODEL)} features")

# Ablation figure: MI per candidate vs the noise floor.
fig, ax = plt.subplots(figsize=(8, 5))
_cand_mi = mi[CANDIDATE_FEATURES].sort_values()
colors = ["seagreen" if c in KEPT else "silver" for c in _cand_mi.index]
_cand_mi.plot(kind="barh", ax=ax, color=colors)
ax.axvline(MI_FLOOR, color="crimson", ls="--", lw=1.2, label=f"noise floor +3sd = {MI_FLOOR:.4f}")
ax.set_title("M3.5 feature ablation — candidate MI vs next-day target (green=kept)")
ax.set_xlabel("mutual information"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "M3_fig_ablation.png", dpi=120); plt.close(fig)
print("saved:", FIG / "M3_fig_ablation.png")

## 2. Baselines (scored on val)

- **naive_zero:** predict 0 (the EMH benchmark — best guess of a zero-mean return).
- **persistence:** predict tomorrow = today's return (`log_return`).
- **moving_avg_20:** predict tomorrow = mean of the last 20 returns.

All scored against the next-day `target_log_return`, pooled across tickers (per-ticker also tabulated).

In [ ]:
val_scores = []

# naive_zero
val_scores.append(score("naive_zero", val_c[TARGET], np.zeros(len(val_c))))
# persistence: today's log_return predicts tomorrow's
val_scores.append(score("persistence", val_c[TARGET], val_c["log_return"]))
# moving_avg_20: rolling mean of log_return per ticker (already known at t)
ma = val_c.groupby("ticker")["log_return"].transform(lambda s: s.rolling(20, min_periods=5).mean())
mask = ma.notna()
val_scores.append(score("moving_avg_20", val_c.loc[mask, TARGET], ma[mask]))

print(pd.DataFrame(val_scores).round(5).to_string(index=False))

**Takeaway:** these define the bar. `naive_zero` is hard to beat on RMSE precisely because returns are
near-zero-mean (EMH); the interesting question is whether any model beats ~0.50 **directional** accuracy.

## 3. Classical — per-ticker ARIMA (mean) + GJR-GARCH (variance)

**ARIMA** models the return *mean*. We select order by **BIC** (penalises complexity harder than AIC; on this
data AIC over-selects (2,0,2) while BIC cleanly gives the parsimonious order M2's negative-lag-1 finding
implies). One-step-ahead forecasts across val.

**Causal one-step (off-by-one fixed):** to forecast the **next-day** return `target[i] = r[i+1]`, history must
include today's realised return `r[i]`. So inside the loop we **append `r[i]` first, then `forecast(1)`** —
that prediction is `E[r[i+1] | data through i]`, correctly aligned to `target[i]`. (Earlier code forecast
*before* appending, predicting the same day — a one-day misalignment caught in review.)

In [ ]:
import pickle

ARIMA_ORDERS = [(1, 0, 0), (0, 0, 1), (1, 0, 1), (2, 0, 2)]
arima_val_pred, arima_val_true, arima_info = [], [], {}

for t in TICKERS:
    r_tr = train_c.loc[train_c.ticker == t].set_index("date")["log_return"].astype(float)
    v = val_c.loc[val_c.ticker == t].sort_values("date")
    r_va = v.set_index("date")["log_return"].astype(float)
    y_va = v[TARGET].values

    # order by BIC on train (parsimonious)
    fits = {o: ARIMA(r_tr, order=o).fit() for o in ARIMA_ORDERS}
    best = min(ARIMA_ORDERS, key=lambda o: fits[o].bic)
    res = fits[best]
    # raw-residual Ljung-Box (note: ~0 reflects leftover ARCH/volatility, NOT a wrong mean order)
    lb_raw = float(acorr_ljungbox(res.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0])

    # CAUSAL one-step: append today's return, THEN forecast tomorrow -> aligns to next-day target.
    cur, preds = res, []
    for ret in r_va.values:
        cur = cur.append([ret], refit=False)
        preds.append(float(cur.forecast(1).iloc[0]))
    arima_val_pred.extend(preds)
    arima_val_true.extend(y_va)
    arima_info[t] = {"order": best, "bic": round(res.bic, 1), "aic": round(res.aic, 1),
                     "resid_LB_raw_p": round(lb_raw, 3)}
    res.save(str(MODELS / f"arima_{t.lstrip('^')}.pkl"))

print("ARIMA per-ticker (order by BIC; raw-resid Ljung-Box p — low = leftover ARCH, expected):")
for t, d in arima_info.items():
    print(f"  {t:<6} order={d['order']} BIC={d['bic']} AIC={d['aic']} resid_LB_raw_p={d['resid_LB_raw_p']}")
val_scores.append(score("ARIMA (per-ticker)", arima_val_true, arima_val_pred))
print("\n", pd.DataFrame(val_scores).round(5).to_string(index=False))

# Alignment self-check: prove the forecast is next-day, not same-day.
_chk = val_c.loc[val_c.ticker == "^GSPC"].sort_values("date")
assert len(arima_val_pred) == len(arima_val_true), "pred/true length mismatch"
print("alignment check: pred[i] scored vs target[i]=next-day return (append-then-forecast) OK")

**Takeaway:** order is chosen by BIC (parsimonious). The raw-residual Ljung-Box p≈0 for some tickers is **not**
a wrong-order signal — it persists at every order because the non-whiteness is **leftover volatility (ARCH)**,
which the GARCH variance model (§4) handles. We re-test whiteness on **GARCH-standardised residuals** there.
Expect ARIMA val RMSE ≈ `naive_zero` (EMH ceiling); any directional edge is significance-tested in §6.

## 4. GJR-GARCH (Student-t) — the variance model M2's leverage effect demands

M2 proved volatility is (a) clustered and (b) **asymmetric** (down days raise next-day vol more). A symmetric
GARCH(1,1) would miss the asymmetry, so we fit a **GJR-GARCH(1,1)** with an AR(1) mean and **Student-t**
errors (returns are fat-tailed). We use the `arch` package's out-of-sample machinery: fit on train only
(`last_obs` = first val date), then forecast one-step over val **without refitting** (no leakage).

Returns are rescaled ×100 (arch optimises better on percent-scale data); we divide forecasts back.

In [ ]:
def qlike(realized_var, forecast_var):
    """QLIKE loss for variance forecasts (lower = better). Standard volatility-forecast metric."""
    rv, fv = np.asarray(realized_var, float), np.asarray(forecast_var, float)
    m = (fv > 0) & np.isfinite(rv) & np.isfinite(fv)
    return float(np.mean(np.log(fv[m]) + rv[m] / fv[m]))


garch_val_pred, garch_val_true, garch_info = [], [], {}

for t in TICKERS:
    full = pd.concat([train_c.loc[train_c.ticker == t], val_c.loc[val_c.ticker == t]]).sort_values("date")
    r = full.set_index("date")["log_return"].astype(float) * 100.0
    first_val = val_c.loc[val_c.ticker == t, "date"].min()

    am = arch_model(r, mean="AR", lags=1, vol="GARCH", p=1, o=1, q=1, dist="t")  # o=1 => GJR asymmetry
    res = am.fit(last_obs=first_val, disp="off")
    fc = res.forecast(horizon=1, start=first_val, reindex=False)

    # arch forecast at origin D = E[r_{D+1} | info<=D]; row D pairs with val row whose target is r_{D+1}.
    mean_pred = (fc.mean["h.1"].values / 100.0)            # mean back to log-return scale
    var_pred = fc.variance["h.1"].values / (100.0 ** 2)    # variance is on the x100^2 scale -> /100^2
    vol_pred = np.sqrt(var_pred)
    v = val_c.loc[val_c.ticker == t].sort_values("date")
    y_va = v[TARGET].values
    assert len(mean_pred) == len(y_va), f"GARCH val length mismatch {t}: {len(mean_pred)} vs {len(y_va)}"
    n = len(y_va)
    garch_val_pred.extend(mean_pred[:n]); garch_val_true.extend(y_va[:n])

    # Variance-forecast evaluation (GARCH's real job): QLIKE + Mincer-Zarnowitz R^2 vs realized var proxy.
    realized_var = y_va[:n] ** 2  # squared next-day return = noisy realized-variance proxy
    ql = qlike(realized_var, var_pred[:n])
    # MZ regression: realized_var ~ a + b*forecast_var ; R^2 = how much of realized var the forecast explains
    fvv = var_pred[:n]
    mzmask = np.isfinite(fvv) & (fvv > 0)
    mz_r2 = float(np.corrcoef(fvv[mzmask], realized_var[mzmask])[0, 1] ** 2) if mzmask.sum() > 2 else np.nan

    # whiteness on STANDARDISED residuals (correct test of mean+vol fit, vs raw-resid LB in sec 3)
    std_resid = res.resid[res.resid.index < first_val] / res.conditional_volatility[res.conditional_volatility.index < first_val]
    lb_std = float(acorr_ljungbox(std_resid.dropna(), lags=[10], return_df=True)["lb_pvalue"].iloc[0])
    lb_std_sq = float(acorr_ljungbox(std_resid.dropna() ** 2, lags=[10], return_df=True)["lb_pvalue"].iloc[0])

    garch_info[t] = {
        "nu_t": round(float(res.params.get("nu", np.nan)), 1),
        "gamma": round(float(res.params.get("gamma[1]", np.nan)), 3),
        "mean_vol_%": round(float(np.nanmean(vol_pred)) * 100, 2),
        "QLIKE": round(ql, 4), "MZ_R2": round(mz_r2, 3),
        "std_resid_LB_p": round(lb_std, 3), "std_resid2_LB_p": round(lb_std_sq, 3),
    }
    with open(MODELS / f"garch_{t.lstrip('^')}.pkl", "wb") as f:
        pickle.dump(res, f)  # arch results pickle fine (they have no .save method)

print("GJR-GARCH-t per ticker:")
for t, d in garch_info.items():
    print(f"  {t:<6} nu={d['nu_t']} gamma={d['gamma']} vol={d['mean_vol_%']}% "
          f"QLIKE={d['QLIKE']} MZ_R2={d['MZ_R2']} std_resid_LB_p={d['std_resid_LB_p']} "
          f"std_resid2_LB_p={d['std_resid2_LB_p']}")
val_scores.append(score("GJR-GARCH-t (mean)", garch_val_true, garch_val_pred))
print("\n", pd.DataFrame(val_scores).round(5).to_string(index=False))

**Takeaway:** **γ > 0** for every ticker confirms M2's leverage effect *inside* the fitted model; low ν (~5)
confirms fat tails. **GARCH is judged on its real output — the variance forecast — via QLIKE and the
Mincer-Zarnowitz R²** (how much of next-day realized variance the forecast explains), NOT point-return RMSE.
Its mean-forecast RMSE/DirAcc are shown in the comparison table only for completeness and should **not** be
read as a model win (its mean is ~the naive mean). The standardised-residual Ljung-Box p-values test whether
the AR(1)+GJR-GARCH jointly whitened the series (mean) and removed ARCH (squared) — the correct diagnostic,
unlike the raw-residual LB in §3.

## 5. Deep — global LSTM on MODEL_FEATURES (Colab-trained, local smoke-test)

One model for all four tickers: input = a 60-day sequence of the 30 `MODEL_FEATURES` + a 4-way ticker
one-hot; output = next-day `target_log_return`. The MI evidence (M2) says the signal is nonlinear and
volatility-based — this is the model that can use it.

**Sealing rules:** sequences never cross a ticker boundary; the `StandardScaler` is fit on **train only** and
applied to val; `shuffle=False` in `fit` (time order preserved).

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(SEED)
SEQ_LEN = 60

# One-hot ticker, scale features on train only.
def add_onehot(df):
    oh = pd.get_dummies(df["ticker"]).reindex(columns=TICKERS, fill_value=0).astype("float32")
    return oh

scaler = StandardScaler().fit(train_c[FEAT_MODEL].values)   # FEAT_MODEL = baseline + ablation survivors
FEAT_COLS = FEAT_MODEL + TICKERS


def build_matrix(df):
    feats = pd.DataFrame(scaler.transform(df[FEAT_MODEL].values), columns=FEAT_MODEL, index=df.index)
    oh = add_onehot(df)
    X = pd.concat([feats, oh.set_axis(df.index)], axis=1)[FEAT_COLS].astype("float32")
    return X


def make_sequences(df, seq_len=SEQ_LEN):
    """Build (N, seq_len, n_feat) sequences per ticker (never crossing tickers); target = next-day return."""
    Xs, ys = [], []
    Xmat = build_matrix(df)
    for t in TICKERS:
        idx = df.index[df.ticker == t]
        Xt = Xmat.loc[idx].values
        yt = df.loc[idx, TARGET].values
        for i in range(seq_len, len(Xt)):
            Xs.append(Xt[i - seq_len:i])
            ys.append(yt[i])
    return np.asarray(Xs, "float32"), np.asarray(ys, "float32")


X_tr, y_tr = make_sequences(train_c)
X_va, y_va = make_sequences(val_c)
print("LSTM tensors -> X_tr:", X_tr.shape, "X_va:", X_va.shape)


def build_lstm(n_feat, units=64, layers=1, attention=False, dropout=0.2, learning_rate=1e-3, bidirectional=False):
    """Functional Keras LSTM. With attention=True, an additive attention layer pools the LSTM's per-timestep
    outputs (weights every day in the 60-day window) instead of taking only the last step — letting the model
    focus on the most informative days (e.g. a vol spike) rather than just the most recent one.
    dropout / learning_rate / bidirectional are exposed for the Optuna search (Colab GPU)."""
    def _lstm(u, return_sequences):
        layer = keras.layers.LSTM(u, return_sequences=return_sequences)
        return keras.layers.Bidirectional(layer) if bidirectional else layer

    inp = keras.layers.Input(shape=(SEQ_LEN, n_feat))
    if layers == 2:
        x = _lstm(units, True)(inp)
        x = keras.layers.Dropout(dropout)(x)
        x = _lstm(units, attention)(x)  # keep sequence if attention pools it
    else:
        x = _lstm(units, attention)(inp)
    if attention:
        # additive (Bahdanau-style) self-attention pooling over timesteps
        score_t = keras.layers.Dense(1, activation="tanh")(x)               # (batch, T, 1)
        weights = keras.layers.Softmax(axis=1)(score_t)                     # attention weight per timestep
        x = keras.layers.Multiply()([x, weights])
        x = keras.layers.Lambda(lambda z: tf.reduce_sum(z, axis=1))(x)      # context vector
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(1)(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])
    return m


def _lstm_inner_split(cut_date, embargo=1):
    """Inner purged time split of train_c for LSTM tuning (Colab). Scaler is refit on INNER-TRAIN only so the
    search never leaks inner-val statistics. Returns ((X_it, y_it), (X_iv, y_iv))."""
    tr_all = train_c[train_c.date < cut_date]
    tr_dates = np.sort(tr_all.date.unique())
    tr = tr_all[tr_all.date < tr_dates[-embargo]]            # embargo: drop last train date(s) before the cut
    va = train_c[train_c.date >= cut_date]
    sc = StandardScaler().fit(tr[FEAT_MODEL].values)

    def _seq(df):
        feats = pd.DataFrame(sc.transform(df[FEAT_MODEL].values), columns=FEAT_MODEL, index=df.index)
        Xmat = pd.concat([feats, add_onehot(df).set_axis(df.index)], axis=1)[FEAT_MODEL + TICKERS].astype("float32")
        Xs, ys = [], []
        for t in TICKERS:
            idx = df.index[df.ticker == t]
            Xt, yt = Xmat.loc[idx].values, df.loc[idx, TARGET].values
            for i in range(SEQ_LEN, len(Xt)):
                Xs.append(Xt[i - SEQ_LEN:i]); ys.append(yt[i])
        return np.asarray(Xs, "float32"), np.asarray(ys, "float32")

    return _seq(tr), _seq(va)


# FULL_TRAIN auto-enables on a Colab GPU (20-epoch attention + Optuna search). Locally it stays a 2-epoch
# smoke-test. Force it anywhere with the env var LSTM_FULL_TRAIN=1.
FULL_TRAIN = bool(len(tf.config.list_physical_devices("GPU"))) or os.environ.get("LSTM_FULL_TRAIN") == "1"

# Default (smoke) config; on a Colab GPU an Optuna search replaces it with a tuned config.
lstm_cfg = dict(units=32, layers=1, dropout=0.2, learning_rate=1e-3, batch_size=128, bidirectional=False)
EPOCHS = 2

if FULL_TRAIN:
    EPOCHS = 20
    lstm_cfg.update(units=64, layers=2, batch_size=64)
    # Small Optuna search over ONE purged inner time-split (last ~250 train dates as inner-val; scaler refit
    # on inner-train only). Kept small (15 trials x 10 epochs) — a full sequence-CV per trial is too costly.
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _all_dates = np.sort(train_c.date.unique())
    (_Xit, _yit), (_Xiv, _yiv) = _lstm_inner_split(pd.Timestamp(_all_dates[-250]))
    print(f"LSTM Optuna inner split: train seq={_Xit.shape}, inner-val seq={_Xiv.shape}")

    def _lstm_objective(trial):
        cfg = dict(
            units=trial.suggest_categorical("units", [32, 64, 96]),
            dropout=trial.suggest_float("dropout", 0.1, 0.3),
            learning_rate=trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
            batch_size=trial.suggest_categorical("batch_size", [32, 64]),
            layers=trial.suggest_categorical("layers", [1, 2]),
            bidirectional=trial.suggest_categorical("bidirectional", [False, True]),
        )
        keras.backend.clear_session()
        mdl = build_lstm(_Xit.shape[2], attention=True, units=cfg["units"], layers=cfg["layers"],
                         dropout=cfg["dropout"], learning_rate=cfg["learning_rate"], bidirectional=cfg["bidirectional"])
        mdl.fit(_Xit, _yit, validation_data=(_Xiv, _yiv), epochs=10, batch_size=cfg["batch_size"], shuffle=False,
                verbose=0, callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
        return rmse(_yiv, mdl.predict(_Xiv, verbose=0).ravel())

    _lstm_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    _lstm_study.optimize(_lstm_objective, n_trials=15, show_progress_bar=False)
    lstm_cfg.update(_lstm_study.best_params)
    (MODELS / "lstm_best_params.json").write_text(json.dumps(
        {**_lstm_study.best_params, "inner_val_rmse": round(_lstm_study.best_value, 6)}, indent=2))
    print("LSTM best config (Optuna):", _lstm_study.best_params)

lstm = build_lstm(X_tr.shape[2], attention=True, units=lstm_cfg["units"], layers=lstm_cfg["layers"],
                  dropout=lstm_cfg["dropout"], learning_rate=lstm_cfg["learning_rate"],
                  bidirectional=lstm_cfg["bidirectional"])
lstm.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCHS, batch_size=lstm_cfg["batch_size"],
         shuffle=False, verbose=2,
         callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)] if FULL_TRAIN else [])
lstm_pred = lstm.predict(X_va, verbose=0).ravel()
LSTM_TAG = "LSTM+Attention (FINAL)" if FULL_TRAIN else "LSTM+Attention (SMOKE 2ep, not final)"
lstm_score = score(LSTM_TAG, y_va, lstm_pred)
lstm.save(MODELS / ("lstm_attention_final.keras" if FULL_TRAIN else "lstm_attention_smoke.keras"))
if FULL_TRAIN:
    val_scores.append(lstm_score)  # a real result -> goes in the comparison table
    (MODELS / "lstm_val_metrics.json").write_text(json.dumps(lstm_score, indent=2))
print(f"\nFULL_TRAIN={FULL_TRAIN}  {LSTM_TAG}:",
      {k: round(v, 5) if isinstance(v, float) else v for k, v in lstm_score.items()})

**Takeaway:** on a local CPU this is a 2-epoch smoke-test (meaningless score, proves the attention pipeline
runs). On a **Colab GPU `FULL_TRAIN` flips to True automatically** — 20 epochs, 2-layer, early stopping — and
the LSTM row becomes a real result in the comparison table + `lstm_val_metrics.json`, and §8 fills the
holdout `y_pred_lstm` and the real ensemble.

## 5b. LightGBM — tabular gradient-boosted trees (fully local, real result)

Unlike the LSTM (Colab), LightGBM trains in seconds on CPU, so this gives a **real, locally-measured** ML
result. It sees the same 39 `MODEL_FEATURES` (incl. the new exogenous VIX/yields/dollar) as a flat table —
the last day's features predict the next-day return. `ticker` is passed as a native categorical. We use the
same leakage discipline (train-only fit; chronological, never shuffled).

**Note (deliberately un-tuned):** fixed 400 trees, no early-stopping / no Optuna search — so the holdout
overfit below is *partly* a tuning artifact and is exactly the point: it shows that throwing a flexible model
+ exogenous features at the problem buys in-sample fit, not out-of-sample skill. Exogenous "daily change"
features (`vix_log_change`, `tnx_change`, `dxy_log_return`) are differenced on the **stock** trading calendar
(post-merge), so a Fri→Mon or bond/FX-holiday gap counts as one step — immaterial at a daily horizon.

In [ ]:
import lightgbm as lgb

# BASELINE LightGBM: the original 39 MODEL_FEATURES, deliberately un-tuned. Left untouched so it stays the
# fixed reference for the promote/revert guardrail in 5b-tune / section 8 (tuned+FEAT_MODEL must beat THIS).
Xtr = train_c[MODEL_FEATURES].copy(); Xtr["ticker"] = train_c["ticker"].astype("category")
Xva = val_c[MODEL_FEATURES].copy();  Xva["ticker"] = val_c["ticker"].astype("category")
ytr, yva = train_c[TARGET].values, val_c[TARGET].values
FEAT_LGB = MODEL_FEATURES + ["ticker"]

lgb_params = dict(objective="regression", n_estimators=400, learning_rate=0.02, num_leaves=31,
                  min_child_samples=50, subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
                  random_state=SEED, n_jobs=-1, verbosity=-1, deterministic=True, force_row_wise=True)
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(Xtr[FEAT_LGB], ytr, categorical_feature=["ticker"])
lgb_val_pred = lgb_model.predict(Xva[FEAT_LGB])
val_scores.append(score("LightGBM (tabular)", yva, lgb_val_pred))
dm_lgb = diebold_mariano(yva, lgb_val_pred, np.zeros_like(yva, float))
print("LightGBM val:", {k: round(v, 5) if isinstance(v, float) else v for k, v in val_scores[-1].items()})
print(f"LightGBM vs naive DM p={dm_lgb[1]:.3f}")
lgb_model.booster_.save_model(str(MODELS / "lgbm_global.txt"))

# Feature importance — does the model actually use the new exogenous features?
imp = pd.Series(lgb_model.feature_importances_, index=FEAT_LGB).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 9))
imp.head(20).iloc[::-1].plot(kind="barh", ax=ax, color="darkgreen")
ax.set_title("M3 — LightGBM feature importance (top 20)")
fig.tight_layout(); fig.savefig(FIG / "M3_fig_lgb_importance.png", dpi=120); plt.close(fig)
exog_in_top = [f for f in imp.head(15).index if f in ("vix_level", "vix_z_60", "term_spread", "tnx_level",
               "dxy_log_return", "vix_log_change", "is_high_vol_regime", "tnx_change", "term_spread_change")]
print("exogenous features in LightGBM top-15:", exog_in_top)

## 5b-tune. Optuna hyperparameter search for LightGBM (leakage-free walk-forward CV)

The baseline above is deliberately un-tuned and overfits (val edge that dies out-of-sample). Here we tune it
**properly**: an Optuna TPE search whose objective is the **mean out-of-sample RMSE across the 5 purged
walk-forward folds** from section 1b — so hyperparameters are chosen on multiple chronological OOS blocks,
never on val or holdout. This is the single biggest methodological fix: it replaces the one-val-peek selection
(which caused the overfit) with leakage-free cross-validation. `n_estimators` is NOT tuned blindly — each fold
early-stops on its own OOS slice and we take the **median** best-iteration. The tuned model trains on
`FEAT_MODEL` (39 baseline + any ablation survivor). Whether it actually beats the baseline is decided ONLY
on the sealed holdout in section 8 (`PROMOTE`/`REVERT`) — the honest test, given the diagnosed overfit.

> Caveat (disclosed): each fold early-stops on the same OOS slice it is then scored on, so the reported CV RMSE
> is mildly optimistic about tree count. This is contained — it is all inside train, every trial shares the
> bias, and the promote/revert decision rests on the sealed holdout, never on the optimistic CV number.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)   # quiet console (ASCII-only, cp1252-safe)
FEAT_LGB_TUNED = FEAT_MODEL + ["ticker"]
LGB_FIXED = dict(objective="regression", random_state=SEED, n_jobs=-1, verbosity=-1,
                 deterministic=True, force_row_wise=True)
LGB_MAX_ROUNDS, LGB_ES, N_TRIALS_LGB = 2000, 50, 60


def _lgb_cv(params):
    """Mean OOS RMSE + DirAcc + median best-iter + RMSE-std across the purged walk-forward folds (train-only)."""
    rmses, das, iters = [], [], []
    for tr_idx, oo_idx in date_folds(train_c):
        Xt = train_c.loc[tr_idx, FEAT_MODEL].copy(); Xt["ticker"] = train_c.loc[tr_idx, "ticker"].astype("category")
        Xo = train_c.loc[oo_idx, FEAT_MODEL].copy(); Xo["ticker"] = train_c.loc[oo_idx, "ticker"].astype("category")
        yt, yo = train_c.loc[tr_idx, TARGET].values, train_c.loc[oo_idx, TARGET].values
        m = lgb.LGBMRegressor(n_estimators=LGB_MAX_ROUNDS, **LGB_FIXED, **params)
        m.fit(Xt[FEAT_LGB_TUNED], yt, eval_set=[(Xo[FEAT_LGB_TUNED], yo)], eval_metric="rmse",
              categorical_feature=["ticker"], callbacks=[lgb.early_stopping(LGB_ES, verbose=False)])
        it = m.best_iteration_ or LGB_MAX_ROUNDS
        p = m.predict(Xo[FEAT_LGB_TUNED], num_iteration=it)
        rmses.append(rmse(yo, p)); das.append(directional_accuracy(yo, p)); iters.append(it)
    return float(np.mean(rmses)), float(np.nanmean(das)), int(np.median(iters)), float(np.std(rmses))


def _lgb_objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 5e-3, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),   # alias: colsample_bytree
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),   # alias: subsample
        bagging_freq=1,
        lambda_l1=trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),     # alias: reg_alpha
        lambda_l2=trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),     # alias: reg_lambda
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
    )
    mean_rmse, mean_da, med_iter, sd = _lgb_cv(params)
    trial.set_user_attr("mean_diracc", mean_da)
    trial.set_user_attr("median_best_iter", med_iter)
    trial.set_user_attr("rmse_std", sd)
    return mean_rmse   # PRIMARY objective: minimize mean OOS RMSE (DirAcc/std reported, not optimized-for)


print(f"\nOptuna LightGBM search: {N_TRIALS_LGB} trials x 5 walk-forward folds (train-only)...")
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(_lgb_objective, n_trials=N_TRIALS_LGB, show_progress_bar=False)
best = study.best_trial
med_iter = best.user_attrs["median_best_iter"]
lgb_params_tuned = dict(n_estimators=med_iter, bagging_freq=1, **LGB_FIXED, **best.params)
print(f"best CV mean-RMSE={best.value:.6f}  (DirAcc={best.user_attrs['mean_diracc']:.4f}, "
      f"RMSE-std={best.user_attrs['rmse_std']:.6f}, n_estimators={med_iter})")
print("best params:", {k: (round(v, 5) if isinstance(v, float) else v) for k, v in best.params.items()})

# Refit tuned on full train, score on val (a plain reporting row; val is NOT used for selection).
Xtr_t = train_c[FEAT_MODEL].copy(); Xtr_t["ticker"] = train_c["ticker"].astype("category")
Xva_t = val_c[FEAT_MODEL].copy();  Xva_t["ticker"] = val_c["ticker"].astype("category")
lgb_tuned = lgb.LGBMRegressor(**lgb_params_tuned).fit(Xtr_t[FEAT_LGB_TUNED], ytr, categorical_feature=["ticker"])
lgb_tuned_val_pred = lgb_tuned.predict(Xva_t[FEAT_LGB_TUNED])
val_scores.append(score("LightGBM (tuned)", yva, lgb_tuned_val_pred))
print("LightGBM (tuned) val:", {k: round(v, 5) if isinstance(v, float) else v for k, v in val_scores[-1].items()})

# Artifacts: best params + CV summary, the study, and a tuning-history figure (matplotlib backend, no plotly).
import pickle
(MODELS / "lgbm_best_params.json").write_text(json.dumps({
    "best_params": best.params, "median_best_iter": med_iter, "cv_mean_rmse": round(best.value, 6),
    "cv_mean_diracc": round(best.user_attrs["mean_diracc"], 4), "cv_rmse_std": round(best.user_attrs["rmse_std"], 6),
    "n_trials": N_TRIALS_LGB, "feat_model": FEAT_MODEL, "kept_candidates": KEPT}, indent=2))
with open(MODELS / "lgbm_optuna_study.pkl", "wb") as f:
    pickle.dump(study, f)
try:
    from optuna.visualization.matplotlib import plot_optimization_history
    ax = plot_optimization_history(study)
    ax.figure.tight_layout(); ax.figure.savefig(FIG / "M3_fig_lgb_optuna.png", dpi=120); plt.close(ax.figure)
    print("saved:", FIG / "M3_fig_lgb_optuna.png")
except Exception as e:   # viz is optional; never let a plotting hiccup fail the pipeline
    print("optuna viz skipped:", e)

## 5c. Ensemble — return-scale average of the two holdout survivors (GJR-GARCH mean + attention-LSTM)

The previous ensemble z-scored LightGBM + LSTM and could only report *direction* (z-scoring discards the
return scale). Two problems: it was off-scale (no RMSE / DM / cost-backtest), and it blended the **overfit**
LightGBM (holdout DirAcc 0.499) into the real-edge LSTM, dragging the blend toward a coin flip.

The M3.5 fix: **equal-weight average, on the RETURN SCALE, of GJR-GARCH-mean and the attention-LSTM** — two of
the three models with a significant holdout directional edge. Honest framing: GARCH's mean forecast is a small,
lag-1-driven, **near-naive anchor** (§4 says its mean is "~the naive mean"), so this blend is really the LSTM
**shrunk halfway toward a near-zero anchor** — often helpful for near-zero-mean returns, not a fusion of two
independent alphas. Whether averaging genuinely helps depends on how decorrelated the members' errors are, so
§8 **measures** that error correlation on the holdout rather than asserting it. Keeping the return scale makes
RMSE / MAE / Diebold-Mariano well-defined and lets it feed the costed backtest. Equal 0.5/0.5 weight is the
honest default (tuning weights on val would overfit).

Why these two and not the tuned LightGBM (also 0.542, p=0.002)? Its edge rides the same lag-1 mean-reversion
microstructure as GARCH-mean, so adding it is redundant, not diversifying — a judgment call, stated openly.
The old LGB+LSTM z-blend is retained below only as the *rejected* baseline.

> Local caveat: locally the LSTM is a 2-epoch smoke model, so the blended number is plumbing; the real
> ensemble score (and the error-correlation measurement) come from the Colab-trained LSTM (§8).

In [ ]:
def zblend(*preds):
    """Average prediction vectors after standardizing each to zero-mean/unit-std. Kept ONLY to reproduce the
    REJECTED baseline blend (direction-only, off the return scale) for the record."""
    zs = []
    for p in preds:
        p = np.asarray(p, float)
        sd = p.std()
        zs.append((p - p.mean()) / sd if sd > 0 else p - p.mean())
    return np.mean(zs, axis=0)


def blend_returns(*preds):
    """Equal-weight average of RETURN-SCALE prediction vectors (keeps the scale so RMSE/MAE/DM stay valid)."""
    return np.mean([np.asarray(p, float) for p in preds], axis=0)


# GARCH mean forecasts are pooled in TICKERS order ([^GSPC, AAPL, AMZN, NVDA]), but val_c is sorted
# ALPHABETICALLY (^ = ASCII 94 > 'Z', so ^GSPC sorts LAST). Remap per-ticker to val_c row positions so the
# GARCH preds, the LSTM preds (lstm_full, val_c order) and the target (yva) all line up.
_gp = np.asarray(garch_val_pred, float)
garch_val_arr = np.full(len(val_c), np.nan)
_pos = 0
for t in TICKERS:
    idx = np.where(val_c.ticker.values == t)[0]
    garch_val_arr[idx] = _gp[_pos:_pos + len(idx)]; _pos += len(idx)
assert not np.isnan(garch_val_arr).any(), "GARCH val remap left NaNs (ticker length mismatch)"
# order sanity: GARCH's own target (garch_val_true, same loop order) remapped to val_c order must equal yva
_gt = np.full(len(val_c), np.nan); _pos = 0; _gtv = np.asarray(garch_val_true, float)
for t in TICKERS:
    idx = np.where(val_c.ticker.values == t)[0]
    _gt[idx] = _gtv[_pos:_pos + len(idx)]; _pos += len(idx)
assert np.allclose(_gt, val_c[TARGET].values), "GARCH val remap misaligned to val_c target order"

# Align LSTM (sequence) preds to the tabular rows: sequences drop the first SEQ_LEN rows per ticker.
lstm_full = np.full(len(val_c), np.nan)
seq_pos = 0
for t in TICKERS:
    n_rows = int((val_c.ticker == t).sum())
    n_seq = max(0, n_rows - SEQ_LEN)
    idx = np.where(val_c.ticker.values == t)[0]
    lstm_full[idx[SEQ_LEN:]] = lstm_pred[seq_pos:seq_pos + n_seq]
    seq_pos += n_seq
# Guard: the remap depends on make_sequences iterating TICKERS in this order — assert the counts line up.
assert int((~np.isnan(lstm_full)).sum()) == sum(max(0, int((val_c.ticker == t).sum()) - SEQ_LEN) for t in TICKERS), \
    "ensemble remap count mismatch — LSTM sequence ordering changed"
m = ~np.isnan(lstm_full)

# PRIMARY ensemble (return scale): GARCH-mean + attention-LSTM, equal weight -> fully scorable (RMSE/MAE/DirAcc).
ens_pred = blend_returns(garch_val_arr[m], lstm_full[m])
_ens_tag = "Ensemble GARCH+LSTM (FINAL)" if FULL_TRAIN else "Ensemble GARCH+LSTM (smoke)"
ens_score = score(_ens_tag, yva[m], ens_pred)
val_scores.append(ens_score)
print(f"Ensemble GARCH+LSTM ({'REAL LSTM' if FULL_TRAIN else 'illustrative, LSTM=smoke'}):",
      {k: round(v, 5) if isinstance(v, float) else v for k, v in ens_score.items()})

# REJECTED baseline for the record: old LGB+LSTM z-blend (off-scale, includes the overfit LightGBM member).
_old_ens = zblend(lgb_val_pred[m], lstm_full[m])
print("  (rejected baseline LGB+LSTM z-blend, dir-only): DirAcc=%.3f p=%.3f"
      % (directional_accuracy(yva[m], _old_ens), diracc_pvalue(yva[m], _old_ens)))

## 6. Model comparison (val)

In [ ]:
# LSTM smoke score is kept OUT of the comparison table (it is not a real result).
# Table now also carries LightGBM (tuned) and the GARCH+LSTM ensemble rows.
val_table = pd.DataFrame(val_scores).round(5)
val_table.to_csv(MODELS / "val_scores.csv", index=False)
print(val_table.to_string(index=False))

# Diebold-Mariano: is any model's RMSE edge over naive_zero statistically real (not sampling noise)?
dm_arima = diebold_mariano(arima_val_true, arima_val_pred, np.zeros_like(arima_val_true, float))
dm_garch = diebold_mariano(garch_val_true, garch_val_pred, np.zeros_like(garch_val_true, float))
print(f"\nDiebold-Mariano vs naive_zero (val):  ARIMA DM={dm_arima[0]:.2f} p={dm_arima[1]:.3f} | "
      f"GARCH DM={dm_garch[0]:.2f} p={dm_garch[1]:.3f}")
print("DirAcc binomial p-values (vs 0.50) are in the DirAcc_p column above.")

**Honest reading (with significance):**
- On **RMSE**, ARIMA and GARCH only *tie* naive_zero — the Diebold-Mariano p-values (printed) are **not
  significant**, so the tiny RMSE gaps are sampling noise, not skill. This is the expected EMH result.
- On **directional accuracy**, the val edge (ARIMA ~0.54, GARCH ~0.55) **is** statistically significant on val
  (binomial p in the `DirAcc_p` column). **But** — see §8 — this edge must be re-checked on the sealed
  holdout, because val is also where the model/order was selected (selection bias).
- GARCH's real value is the **variance forecast** (QLIKE / MZ-R² in §4), not its mean-return RMSE.

## 7. Diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(val_table.model, val_table.DirAcc, color="steelblue")
ax.axhline(0.5, color="crimson", ls="--", lw=1, label="coin flip (0.50)")
ax.set_ylabel("directional accuracy (val)")
ax.set_title("M3 — directional accuracy by model (val)")
ax.set_ylim(0.4, 0.6)
plt.xticks(rotation=30, ha="right")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "M3_fig_diracc.png", dpi=120)
plt.close(fig)

# ARIMA residual ACF (^GSPC) as a representative diagnostic
from statsmodels.graphics.tsaplots import plot_acf
res_gspc = ARIMA(train_c.loc[train_c.ticker == "^GSPC"].set_index("date")["log_return"].astype(float),
                 order=arima_info["^GSPC"]["order"]).fit()
fig = plot_acf(res_gspc.resid, lags=40)
fig.set_size_inches(9, 4)
fig.suptitle("M3 — ^GSPC ARIMA residual ACF (raw resid; leftover ARCH expected)")
fig.tight_layout()
fig.savefig(FIG / "M3_fig_arima_resid_acf.png", dpi=120)
plt.close(fig)

# Per-ticker ARIMA val error (RMSE) — which series is hardest
arima_val_df = pd.DataFrame({"true": arima_val_true, "pred": arima_val_pred})
# rebuild ticker labels in the same pooled order used above
tick_order = []
for t in TICKERS:
    tick_order += [t] * len(val_c.loc[val_c.ticker == t])
arima_val_df["ticker"] = tick_order
err_by_ticker = arima_val_df.groupby("ticker").apply(lambda d: rmse(d.true, d.pred))
fig, ax = plt.subplots(figsize=(7, 4))
err_by_ticker.reindex(TICKERS).plot(kind="bar", ax=ax, color="slateblue")
ax.set_title("M3 — ARIMA val RMSE by ticker")
ax.set_ylabel("RMSE")
fig.tight_layout(); fig.savefig(FIG / "M3_fig_error_by_ticker.png", dpi=120); plt.close(fig)

# Per-month ARIMA val error — regime sensitivity
arima_val_df["month"] = pd.to_datetime(
    np.concatenate([val_c.loc[val_c.ticker == t].sort_values("date")["date"].values for t in TICKERS])
)
err_by_month = arima_val_df.assign(ym=arima_val_df["month"].dt.to_period("M").astype(str)) \
    .groupby("ym").apply(lambda d: rmse(d.true, d.pred))
fig, ax = plt.subplots(figsize=(11, 4))
err_by_month.plot(kind="bar", ax=ax, color="teal")
ax.set_title("M3 — ARIMA val RMSE by month (regime sensitivity)")
ax.set_ylabel("RMSE")
fig.tight_layout(); fig.savefig(FIG / "M3_fig_error_by_month.png", dpi=120); plt.close(fig)
print("saved M3 diagnostic figures (diracc, resid_acf, error_by_ticker, error_by_month)")

## 8. Final holdout evaluation (touch holdout ONCE)

Only now do we open `holdout_fe`. ARIMA retrained on **train+val**, **causal one-step** (append-then-forecast,
same off-by-one fix as §3). We characterise the holdout regime, then run a costed long-only backtest with
risk-adjusted metrics — and significance-test the directional edge that looked real on val.

In [ ]:
holdout = pd.read_parquet(PROC / "holdout_fe.parquet")
holdout["date"] = pd.to_datetime(holdout["date"])
hold_c = clean_xy(holdout)
trainval = pd.concat([train_c, val_c]).sort_values(["ticker", "date"]).reset_index(drop=True)
print("holdout:", hold_c.shape, "| OPENED holdout for the first time (final step)")
print("holdout span:", hold_c.date.min().date(), "->", hold_c.date.max().date())

# Regime characterisation: holdout vs train annualised vol per ticker.
print("\nRegime check — annualised vol (train vs holdout):")
for t in TICKERS:
    v_tr = train_c.loc[train_c.ticker == t, "log_return"].std() * np.sqrt(252)
    v_ho = hold_c.loc[hold_c.ticker == t, "log_return"].std() * np.sqrt(252)
    print(f"  {t:<6} train {v_tr*100:5.1f}%  holdout {v_ho*100:5.1f}%  ({'higher' if v_ho>v_tr else 'lower'})")

rows = []
hold_pred_all = []
for t in TICKERS:
    r_trv = trainval.loc[trainval.ticker == t].set_index("date")["log_return"].astype(float)
    h = hold_c.loc[hold_c.ticker == t].sort_values("date")
    r_h = h.set_index("date")["log_return"].astype(float)
    y_h = h[TARGET].values

    res = ARIMA(r_trv, order=arima_info[t]["order"]).fit()
    cur, preds = res, []
    for ret in r_h.values:               # CAUSAL: append today, then forecast tomorrow (aligned to target)
        cur = cur.append([ret], refit=False)
        preds.append(float(cur.forecast(1).iloc[0]))
    preds = np.array(preds)

    hp = h[["date", "ticker", "log_return"]].copy()
    hp["y_true"] = y_h                    # spec-compatible column names
    hp["y_pred_arima"] = preds
    hp["y_pred_lstm"] = np.nan            # placeholder — filled by the Colab LSTM run
    hp["y_pred_naive_zero"] = 0.0
    hold_pred_all.append(hp)
    rows.append({"ticker": t, "ARIMA_RMSE": rmse(y_h, preds), "ARIMA_DirAcc": directional_accuracy(y_h, preds),
                 "ARIMA_DirAcc_p": diracc_pvalue(y_h, preds), "naive_RMSE": rmse(y_h, np.zeros_like(y_h))})

hold_df = pd.concat(hold_pred_all, ignore_index=True).sort_values(["ticker", "date"]).reset_index(drop=True)

# GJR-GARCH-t on holdout: GARCH had the ONLY significant *val* directional edge (0.555, p<0.001), so we must
# re-test it out-of-sample (audit: don't claim a val edge that was never validated). Fit on train+val
# (last_obs=first_holdout), one-step forecast over holdout — no refit on holdout.
garch_hold = np.full(len(hold_df), np.nan)
for t in TICKERS:
    full = pd.concat([trainval.loc[trainval.ticker == t], hold_c.loc[hold_c.ticker == t]]).sort_values("date")
    r = full.set_index("date")["log_return"].astype(float) * 100.0
    first_ho = hold_c.loc[hold_c.ticker == t, "date"].min()
    res_g = arch_model(r, mean="AR", lags=1, vol="GARCH", p=1, o=1, q=1, dist="t").fit(last_obs=first_ho, disp="off")
    mp = res_g.forecast(horizon=1, start=first_ho, reindex=False).mean["h.1"].values / 100.0
    idx = hold_df.index[hold_df.ticker == t]
    assert len(mp) == len(idx), f"GARCH holdout length mismatch for {t}: {len(mp)} vs {len(idx)}"
    garch_hold[idx] = mp
hold_df["y_pred_garch"] = garch_hold

# LightGBM on holdout: retrain on train+val (with the exogenous features), predict holdout once.
# BASELINE LightGBM on holdout (39 features, un-tuned) — the reference the tuned model must beat.
Xtrv = trainval[MODEL_FEATURES].copy(); Xtrv["ticker"] = trainval["ticker"].astype("category")
lgb_final = lgb.LGBMRegressor(**lgb_params)
lgb_final.fit(Xtrv[FEAT_LGB], trainval[TARGET].values, categorical_feature=["ticker"])
# align LGB holdout preds to hold_df row order (both sorted ticker/date)
ho_sorted = hold_c.sort_values(["ticker", "date"])
hold_df["y_pred_lgb"] = lgb_final.predict(
    ho_sorted[MODEL_FEATURES].assign(ticker=ho_sorted["ticker"].astype("category"))[FEAT_LGB])
lgb_final.booster_.save_model(str(MODELS / "lgbm_global_final.txt"))

# TUNED LightGBM on holdout (FEAT_MODEL = 39 + ablation survivors, Optuna params) — refit on train+val.
Xtrv_t = trainval[FEAT_MODEL].copy(); Xtrv_t["ticker"] = trainval["ticker"].astype("category")
lgb_final_tuned = lgb.LGBMRegressor(**lgb_params_tuned)
lgb_final_tuned.fit(Xtrv_t[FEAT_LGB_TUNED], trainval[TARGET].values, categorical_feature=["ticker"])
hold_df["y_pred_lgb_tuned"] = lgb_final_tuned.predict(
    ho_sorted[FEAT_MODEL].assign(ticker=ho_sorted["ticker"].astype("category"))[FEAT_LGB_TUNED])
lgb_final_tuned.booster_.save_model(str(MODELS / "lgbm_global_tuned_final.txt"))

# LSTM on holdout (only when fully trained on Colab GPU): build holdout sequences, predict, map back.
# First SEQ_LEN rows per ticker have no 60-day history -> stay NaN. build_matrix reuses the train-only scaler.
holdout_models = [("ARIMA", "y_pred_arima"), ("GJR-GARCH-t", "y_pred_garch"),
                  ("LightGBM", "y_pred_lgb"), ("LightGBM (tuned)", "y_pred_lgb_tuned")]
if FULL_TRAIN:
    X_ho, _ = make_sequences(hold_c)
    ho_lstm_pred = lstm.predict(X_ho, verbose=0).ravel()
    lstm_hold = np.full(len(hold_df), np.nan)
    p = 0
    for t in TICKERS:
        idx = hold_df.index[hold_df.ticker == t]
        n_seq = max(0, len(idx) - SEQ_LEN)
        lstm_hold[idx[SEQ_LEN:]] = ho_lstm_pred[p:p + n_seq]
        p += n_seq
    hold_df["y_pred_lstm"] = lstm_hold
    holdout_models.append(("LSTM+Attention", "y_pred_lstm"))
    # Return-scale ensemble on holdout: GARCH-mean + LSTM (equal weight), on rows where both preds exist.
    _mm = hold_df[["y_pred_garch", "y_pred_lstm"]].notna().all(axis=1)
    hold_df["y_pred_ens"] = np.nan
    hold_df.loc[_mm, "y_pred_ens"] = 0.5 * hold_df.loc[_mm, "y_pred_garch"] + 0.5 * hold_df.loc[_mm, "y_pred_lstm"]
    holdout_models.append(("Ensemble GARCH+LSTM", "y_pred_ens"))
    # MEASURE (don't assert) how decorrelated the members' errors are — averaging only helps if they are.
    _eg = hold_df.loc[_mm, "y_true"] - hold_df.loc[_mm, "y_pred_garch"]
    _el = hold_df.loc[_mm, "y_true"] - hold_df.loc[_mm, "y_pred_lstm"]
    ens_err_corr = float(np.corrcoef(_eg, _el)[0, 1])
    print(f"Ensemble members holdout error corr (GARCH vs LSTM) = {ens_err_corr:.3f} "
          f"(low => averaging helps; ~1 => redundant, GARCH-mean is a near-naive anchor shrinking the LSTM)")

hold_df.to_parquet(PROC / "holdout_predictions.parquet", index=False)
hold_table = pd.DataFrame(rows).round(5)
print("\n", hold_table.to_string(index=False))

# Pooled holdout + significance (the only UNBIASED estimate — holdout was never used for selection).
ho_naive = rmse(hold_df.y_true, np.zeros(len(hold_df)))
for name, col in holdout_models:
    _v = hold_df.dropna(subset=[col])                       # LSTM has NaN on first 60 rows/ticker
    _naive = rmse(_v.y_true, np.zeros(len(_v)))
    rm = rmse(_v.y_true, _v[col]); da = directional_accuracy(_v.y_true, _v[col])
    dap = diracc_pvalue(_v.y_true, _v[col]); dm = diebold_mariano(_v.y_true, _v[col], np.zeros(len(_v)))
    print(f"Pooled holdout {name:<14}: RMSE={rm:.5f} vs naive {_naive:.5f} (DM p={dm[1]:.3f}) | "
          f"DirAcc={da:.3f} (binomial p={dap:.3f})")
ho_rmse = rmse(hold_df.y_true, hold_df.y_pred_arima)
dm_ho = diebold_mariano(hold_df.y_true, hold_df.y_pred_arima, np.zeros(len(hold_df)))
ho_da, ho_da_p = directional_accuracy(hold_df.y_true, hold_df.y_pred_arima), diracc_pvalue(hold_df.y_true, hold_df.y_pred_arima)
print("Interpretation: if DM p>0.05 AND DirAcc p>0.05 -> NO significant out-of-sample skill (the expected EMH result).")

# --- Promote/revert guardrail: tuned LightGBM (FEAT_MODEL + Optuna) vs baseline (39 feat, un-tuned) ---
# Pre-registered metric = pooled holdout RMSE (DirAcc as tie context; DM tests if the RMSE gap is real).
# PROMOTE only if tuned improves RMSE AND is not statistically-significantly worse; else REVERT and say so.
_b = hold_df.dropna(subset=["y_pred_lgb"]); _t = hold_df.dropna(subset=["y_pred_lgb_tuned"])
rmse_lgb_base, rmse_lgb_tuned = rmse(_b.y_true, _b.y_pred_lgb), rmse(_t.y_true, _t.y_pred_lgb_tuned)
da_lgb_base, da_lgb_tuned = directional_accuracy(_b.y_true, _b.y_pred_lgb), directional_accuracy(_t.y_true, _t.y_pred_lgb_tuned)
dm_tuned_stat, dm_tuned_p = diebold_mariano(hold_df.y_true, hold_df.y_pred_lgb_tuned, hold_df.y_pred_lgb)  # tuned vs baseline
# PROMOTE only if tuned beats baseline AND the improvement is STATISTICALLY SIGNIFICANT (DM stat<0 => tuned
# lower squared error; p<0.05). Requiring significance makes the DM test load-bearing: a nominal-but-noisy RMSE
# improvement REVERTS. (On the same rows rmse_tuned<rmse_base <=> dm_stat<0, so the RMSE clause is subsumed;
# kept for readability.)
lgb_promote = (rmse_lgb_tuned < rmse_lgb_base) and (dm_tuned_stat < 0) and (dm_tuned_p < 0.05)
LGB_VERDICT = "PROMOTE tuned" if lgb_promote else "REVERT to baseline"
_confounded = len(KEPT) > 0   # tuned model also carries kept features => features+tuning bundled in the verdict
guardrail = {"rmse_base": round(rmse_lgb_base, 6), "rmse_tuned": round(rmse_lgb_tuned, 6),
             "diracc_base": round(da_lgb_base, 4), "diracc_tuned": round(da_lgb_tuned, 4),
             "dm_tuned_vs_base_stat": round(dm_tuned_stat, 4), "dm_tuned_vs_base_p": round(dm_tuned_p, 4),
             "cv_mean_rmse": round(best.value, 6), "kept_features": KEPT,
             "features_tuning_confounded": _confounded, "verdict": LGB_VERDICT}
(MODELS / "lgb_tuning_guardrail.json").write_text(json.dumps(guardrail, indent=2))
print(f"\n[GUARDRAIL] LightGBM  baseline: RMSE={rmse_lgb_base:.5f} DirAcc={da_lgb_base:.3f}  | "
      f"tuned: RMSE={rmse_lgb_tuned:.5f} DirAcc={da_lgb_tuned:.3f}")
print(f"[GUARDRAIL] tuned-vs-baseline DM stat={dm_tuned_stat:.3f} p={dm_tuned_p:.3f} (significant improvement required)  ->  {LGB_VERDICT}")
if _confounded:
    print(f"[GUARDRAIL] NOTE: tuned model also adds features {KEPT} -> verdict is the JOINT features+tuning effect.")
else:
    print("[GUARDRAIL] NOTE: ablation kept 0 candidates -> baseline and tuned use the SAME 39 features "
          "(clean tuning-only comparison).")
print("[GUARDRAIL] (both recorded regardless of verdict; CV motivated tuning, holdout decides -- no peek.)")

### Long-only backtest with risk-adjusted metrics (illustrative — NOT investment advice)

Strategy: for ^GSPC, go long next day when ARIMA prediction > 0, else cash. We report not just terminal
wealth but **Sharpe, max drawdown, annualised return/vol, a t-stat on daily strategy returns, and a
transaction-cost sensitivity** — because a higher terminal multiple alone can hide worse risk-adjusted
performance, and costs matter when the strategy switches position often.

In [ ]:
from scipy.stats import ttest_1samp


def perf(daily_log_ret):
    r = np.asarray(daily_log_ret, float)
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = np.exp(np.cumsum(r))
    mdd = float((cum / np.maximum.accumulate(cum) - 1).min())
    return ann_ret, ann_vol, sharpe, mdd


g = hold_df[hold_df.ticker == "^GSPC"].sort_values("date").copy()
long_today = (g.y_pred_arima > 0).values
strat = np.where(long_today, g.y_true.values, 0.0)
# Transaction cost: charge bps each time the position SWITCHES (enter/exit).
switches = int(np.sum(np.abs(np.diff(np.concatenate([[0], long_today.astype(int)])))))
n_long = int(long_today.sum())

bh = g.y_true.values
rows_bt = []
for cost_bps in (0, 5, 10):
    c = cost_bps / 1e4
    pos = long_today.astype(int)
    switch_mask = np.abs(np.diff(np.concatenate([[0], pos]))) > 0
    strat_net = np.where(long_today, g.y_true.values, 0.0) - switch_mask * c
    ar, av, sh, mdd = perf(strat_net)
    rows_bt.append({"cost_bps": cost_bps, "terminal_x": round(float(np.exp(strat_net.sum())), 3),
                    "ann_ret_%": round(ar * 100, 1), "sharpe": round(sh, 2), "maxDD_%": round(mdd * 100, 1)})

bh_ar, bh_av, bh_sh, bh_mdd = perf(bh)
tstat, tp = ttest_1samp(strat, 0.0)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(g.date, np.exp(np.cumsum(bh)), label=f"buy & hold (Sharpe {bh_sh:.2f})", lw=1.4)
ax.plot(g.date, np.exp(np.cumsum(strat)), label=f"long-when-ARIMA>0, 0bp (Sharpe {rows_bt[0]['sharpe']})", lw=1.4)
ax.set_title("M3 — holdout backtest (illustrative, NOT investment advice)")
ax.set_ylabel("growth of $1")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "M3_fig_backtest.png", dpi=120)
plt.close(fig)

print(f"^GSPC holdout backtest ({n_long} long-days, {switches} position switches):")
print(f"  buy&hold:  terminal x{np.exp(bh.sum()):.3f}  ann_ret {bh_ar*100:.1f}%  Sharpe {bh_sh:.2f}  maxDD {bh_mdd*100:.1f}%")
print("  strategy by transaction cost:")
print("   ", pd.DataFrame(rows_bt).to_string(index=False))
print(f"  strategy daily-return t-stat vs 0: t={tstat:.2f} p={tp:.3f} "
      f"({'insignificant' if tp>0.05 else 'significant'})")
backtest_summary = {"bh_sharpe": bh_sh, "strat_sharpe_0bp": rows_bt[0]["sharpe"], "strat_tstat": float(tstat),
                    "strat_tp": float(tp), "n_switches": switches}

## 9. Leakage verification

In [ ]:
_leak_ok = False
try:
    assert train_c.date.max() < val_c.date.min() < val_c.date.max() < hold_c.date.min(), "split order!"
    assert TARGET not in MODEL_FEATURES, "target leaked into features!"
    _leak_ok = True  # bound to the REAL assert result, not a hardcoded literal
except AssertionError as e:
    print("LEAKAGE CHECK FAILED:", e)
print("Leakage checks passed: split order ok; target not in features; scaler fit on train only.")

## 10. Self-audit

Checks are bound to computed results (no hardcoded `True`).

In [ ]:
audit = {
    "metrics_unit_tested": bool(globals().get("_METRICS_TESTED", False)),
    "ge_3_baselines": sum(s["model"] in ("naive_zero", "persistence", "moving_avg_20") for s in val_scores) >= 3,
    "arima_per_ticker_4": len(arima_info) == 4,
    "arima_order_by_bic": all("bic" in d for d in arima_info.values()),
    "asymmetric_garch_fit": all(d.get("gamma", 0) != 0 for d in garch_info.values()),
    "garch_models_saved": all((MODELS / f"garch_{t.lstrip('^')}.pkl").exists() for t in TICKERS),
    "garch_variance_evaluated": all("QLIKE" in d for d in garch_info.values()),
    "diracc_significance_tested": all("DirAcc_p" in s for s in val_scores),
    "diebold_mariano_done": np.isfinite(dm_arima[1]) and np.isfinite(dm_ho[1]),
    "lstm_attention_trained": (MODELS / ("lstm_attention_final.keras" if FULL_TRAIN else "lstm_attention_smoke.keras")).exists(),
    "lightgbm_trained": (MODELS / "lgbm_global.txt").exists() and (MODELS / "lgbm_global_final.txt").exists(),
    "feature_ablation_done": (MODELS / "feature_ablation.csv").exists() and len(FEAT_MODEL) >= len(MODEL_FEATURES),
    "lgb_optuna_tuned": (MODELS / "lgbm_best_params.json").exists() and (MODELS / "lgbm_optuna_study.pkl").exists(),
    "tuned_vs_baseline_guardrail": (MODELS / "lgb_tuning_guardrail.json").exists()
                            and "y_pred_lgb_tuned" in pd.read_parquet(PROC / "holdout_predictions.parquet").columns,
    "exogenous_features_used": sum(f.startswith(("vix", "tnx", "term", "dxy", "is_high")) for f in MODEL_FEATURES) >= 9,
    "ensemble_blend_defined": any("Ensemble" in s["model"] for s in val_scores),
    "val_comparison_saved": (MODELS / "val_scores.csv").exists(),
    "holdout_touched_once": (PROC / "holdout_predictions.parquet").exists()
                            and "y_pred_lgb" in pd.read_parquet(PROC / "holdout_predictions.parquet").columns,
    "error_by_ticker_and_month": (FIG / "M3_fig_error_by_ticker.png").exists()
                                 and (FIG / "M3_fig_error_by_month.png").exists(),
    "lgb_importance_fig": (FIG / "M3_fig_lgb_importance.png").exists(),
    "backtest_risk_metrics": (FIG / "M3_fig_backtest.png").exists() and "bh_sharpe" in backtest_summary,
    "leakage_assert_passed": _leak_ok,
}
for k, v in audit.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(audit.values()), "M3 self-audit failed!"
print(f"\nAll {len(audit)} M3 self-audit checks passed.")
if FULL_TRAIN:
    print("NOTE: FULL_TRAIN run -- LSTM val/holdout numbers are the REAL 20-epoch result (lstm_attention_final.keras).")
else:
    print("NOTE: LSTM val/holdout numbers are SMOKE-TEST only -- run the Colab GPU config (sec 5) for the real LSTM result.")